In [2]:
import json
import os

from dotenv import load_dotenv
from groq import Groq
load_dotenv("../.env")

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)


In [10]:
import sys

sys.path.append("../backend")

from app.services.llm import generate_response

In [11]:
response = generate_response([
    {
        "role": "user",
        "content": "Say hello in one sentence."
    }
])

print(response)

Hello! I hope you're having a wonderful day.


In [2]:
import sys

sys.path.append("../backend")

from app.services.interview_logic import (
    should_end_interview,
    should_follow_up,
    choose_next_topic,
    determine_answer_state,
    can_follow_up,
    adjust_difficulty
)

print("Interview logic imported successfully!")

Interview logic imported successfully!


In [3]:
print(adjust_difficulty("easy", "strong"))
print(adjust_difficulty("medium", "strong"))
print(adjust_difficulty("hard", "strong"))

print(adjust_difficulty("hard", "technical_gap"))
print(adjust_difficulty("medium", "shallow"))

medium
hard
hard
medium
easy


In [5]:
import sys

if "../backend" not in sys.path:
    sys.path.append("../backend")

from app.agents.manager import manager_decision

print("Manager imported successfully!")

Manager imported successfully!


In [1]:
import sys

if "../backend" not in sys.path:
    sys.path.append("../backend")

from app.prompts import load_prompt

prompt = load_prompt("interviewer.txt")

print(prompt[:500])

INTERVIEWER_PROMPT = """
You are the interviewer in a realistic technical job interview.

Your job is to communicate naturally with the candidate.

You receive:
- candidate profile
- target role
- manager decision
- generated question
- previous conversation

Rules:

1. Ask exactly one question at a time.
2. Be professional but conversational.
3. Do not give the candidate the answer.
4. Do not evaluate the candidate.
5. Do not reveal internal scores or agent decisions.
6. If the action is a foll


In [15]:
from app.agents.interviewer import interviewer_agent

In [16]:
test_decision = {
    "action": "ask_question",
    "topic": "Machine Learning",
    "difficulty": "medium",
    "reason": "Testing interviewer"
}

test_question = {
    "question": "Why did you choose XGBoost for your project?",
    "topic": "Machine Learning",
    "difficulty": "medium",
    "question_type": "technical",
    "expected_concepts": [
        "model selection",
        "tabular data",
        "nonlinear relationships"
    ]
}

test_candidate = {
    "name": "Candidate",
    "skills": [
        "Python",
        "Machine Learning",
        "Scikit-learn"
    ],
    "projects": [
        "Machine Learning Project"
    ]
}

In [17]:
response = interviewer_agent(
    decision=test_decision,
    question=test_question,
    candidate_profile=test_candidate,
    conversation_history=[]
)

print("MESSAGE:")
print(response.message)

print("\nQUESTION:")
print(response.question)

MESSAGE:
I see you worked on a Machine Learning project using Python and scikit‑learn. I'd like to dive a bit deeper into your design choices.

QUESTION:
Why did you choose XGBoost for your project?


In [18]:
from app.agents.evaluator import evaluate_answer

In [19]:
test_question = {
    "question": "Why did you choose XGBoost for your project?",
    "topic": "Machine Learning",
    "category": "Model Selection",
    "difficulty": "medium",
    "question_type": "technical",
    "expected_concepts": [
        "model selection",
        "tabular data",
        "nonlinear relationships"
    ]
}

test_answer = """
I chose XGBoost because it generally performs well on
structured tabular data and can capture nonlinear
relationships. I also wanted a model that could handle
feature interactions effectively.
"""

In [20]:
evaluation = evaluate_answer(
    question=test_question,
    candidate_answer=test_answer,
    target_role="Data Scientist"
)

print(evaluation.model_dump())

{'overall_score': 8.0, 'technical_accuracy': 9.0, 'depth': 6.0, 'reasoning': 6.0, 'clarity': 8.0, 'communication': 8.0, 'confidence': 8.0, 'strengths': ["Correctly identifies XGBoost's strength on structured tabular data", 'Mentions ability to capture nonlinear relationships', 'Notes handling of feature interactions', 'Clear and concise language'], 'weaknesses': ['Answer is brief and lacks discussion of other XGBoost advantages such as regularization, speed, and handling of missing values', 'Does not explain the model selection process or why XGBoost was chosen over alternative algorithms'], 'should_challenge': True, 'suggested_follow_up': 'Can you elaborate on the specific reasons you selected XGBoost over other models for this project, including how its regularization, handling of missing values, and hyperparameter tuning influenced your decision?', 'missing_concepts': ['model selection']}


In [46]:
import importlib
import app.models.interview

importlib.reload(app.models.interview)

<module 'app.models.interview' from 'c:\\Users\\USER\\Desktop\\vs files\\AIML_Projects\\InterviewHive\\notebooks\\../backend\\app\\models\\interview.py'>

In [41]:
from app.models.interview import SkepticResponse
from app.agents.skeptic import skeptic_agent
print(SkepticResponse)

<class 'app.models.interview.SkepticResponse'>


In [37]:
test_question = {
    "question": "How did you improve the performance of your machine learning system?",
    "topic": "Machine Learning",
    "difficulty": "medium",
    "question_type": "technical"
}

test_answer = """
I improved the model performance by 80% and made the
system much faster.
"""

test_candidate = {
    "skills": [
        "Python",
        "Machine Learning",
        "Scikit-learn"
    ],
    "projects": [
        "Machine Learning Project"
    ]
}

In [38]:
result = skeptic_agent(
    candidate_answer=test_answer,
    question=test_question,
    candidate_profile=test_candidate
)

print(result.model_dump())

{'should_challenge': True, 'concern': 'The candidate claims an 80% improvement in model performance and increased system speed without providing any context such as baseline metrics, evaluation methodology, dataset details, or their specific contribution to the improvement.', 'challenge_question': 'Can you describe the baseline performance you started from, the metric you used to measure the 80% improvement, the dataset and experimental setup, the specific changes you made (e.g., feature engineering, model architecture, hyperparameter tuning), and how you measured the speed gains?'}


In [42]:
from app.agents.judge import judge_agent

In [43]:
test_questions = [
    {
        "question": "Why did you choose XGBoost?",
        "topic": "Machine Learning",
        "difficulty": "medium"
    }
]

test_answers = [
    "I chose XGBoost because it performs well on structured data."
]

test_evaluations = [
    {
        "overall_score": 7,
        "technical_accuracy": 8,
        "depth": 6,
        "reasoning": 7,
        "clarity": 8,
        "communication": 8,
        "confidence": 7,
        "strengths": ["Clear explanation"],
        "weaknesses": ["Limited depth"],
        "should_challenge": False,
        "suggested_follow_up": "",
        "missing_concepts": []
    }
]

In [44]:
report = judge_agent(
    candidate_profile=test_candidate,
    target_role="Data Scientist",
    questions=test_questions,
    answers=test_answers,
    evaluations=test_evaluations,
    topics_covered=["Machine Learning"]
)

print(report.model_dump())

{'overall_score': 6.0, 'technical_score': 7.0, 'problem_solving_score': 5.0, 'communication_score': 8.0, 'confidence_score': 7.0, 'depth_score': 5.0, 'strengths': ['Clear and concise explanation', 'Accurate high‑level claim about XGBoost performance on structured data', 'Good communication style and confidence'], 'weaknesses': ['Very limited depth; did not discuss why XGBoost outperforms other models (e.g., gradient boosting mechanics, regularization, handling of missing values)', 'No mention of model evaluation, hyper‑parameter tuning, or potential drawbacks', 'Did not demonstrate problem‑solving or reasoning beyond a generic statement'], 'red_flags': [], 'recommended_topics': ['XGBoost internals (gradient boosting, regularization, tree pruning)', 'Model evaluation metrics and validation strategies', 'Hyper‑parameter tuning for tree‑based models', 'Feature engineering for structured data', 'Comparative analysis of XGBoost vs. other algorithms (Random Forest, LightGBM, CatBoost)'], 'su

In [56]:
import importlib
import app.services.interview_logic as interview_logic

importlib.reload(interview_logic)

from app.services.interview_logic import (
    determine_answer_state,
    adjust_difficulty,
    check_question_history
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [57]:
evaluation = {
    "technical_accuracy": 9,
    "depth": 8,
    "reasoning": 8,
    "overall_score": 9
}

print(
    determine_answer_state(evaluation)
)

strong


In [58]:
print(
    adjust_difficulty(
        "medium",
        "strong"
    )
)

hard


In [59]:
questions = [
    "What is overfitting in machine learning?",
    "Explain the concept of regularization."
]

print(
    check_question_history(
        "What is overfitting in ML?",
        questions
    )
)

{'is_duplicate': True, 'similarity': 0.8391910195350647, 'matched_question': 'What is overfitting in machine learning?'}
